# InsightFuel Analytics — 01. Ingestão (Bronze)

**Objetivo:** realizar a primeira etapa do pipeline, carregando os arquivos brutos das fontes ANP e IBGE e registrando-os na camada **Bronze**.

### Fontes esperadas

- **ANP:** 6 arquivos CSV, referentes aos 1º e 2º semestres de 2021, 2022 e 2023.
- **IBGE População:** `populacao_raw.xlsx`
- **IBGE PIB:** `pib_raw.xlsx`

### Saídas

Os arquivos originais permanecem em `data/bronze/` e são carregados no DuckDB nas tabelas:

- `bronze.anp_raw`
- `bronze.populacao_raw`
- `bronze.pib_raw`
- `bronze.ingestion_log`


## 1. Configuração do projeto

O notebook procura automaticamente a raiz do projeto a partir da pasta `data/bronze`. Isso permite executá-lo tanto a partir da raiz do projeto quanto da pasta `notebooks`.


In [1]:
from pathlib import Path
import re
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start] + list(start.parents):
        if (path / "data" / "bronze").exists():
            return path
    raise FileNotFoundError(
        "Não encontrei a pasta data/bronze. "
        "Abra o notebook dentro da estrutura do projeto InsightFuel."
    )

PROJECT_ROOT = find_project_root(Path.cwd())

BRONZE_DIR = PROJECT_ROOT / "data" / "bronze"
ANP_DIR = BRONZE_DIR / "anp"
POP_DIR = BRONZE_DIR / "ibge_populacao"
PIB_DIR = BRONZE_DIR / "ibge_pib"

DB_DIR = PROJECT_ROOT / "database"
DB_PATH = DB_DIR / "insightfuel.duckdb"
DB_DIR.mkdir(parents=True, exist_ok=True)

print("Projeto:", PROJECT_ROOT)
print("Bronze:", BRONZE_DIR)
print("DuckDB:", DB_PATH)


Projeto: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento
Bronze: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento\data\bronze
DuckDB: C:\Users\Bruna\anaconda_projects\Trabalho_Armazenamento\database\insightfuel.duckdb


## 2. Verificação dos arquivos de entrada

In [2]:
expected_anp = [
    "anp_2021_1_raw.csv",
    "anp_2021_2_raw.csv",
    "anp_2022_1_raw.csv",
    "anp_2022_2_raw.csv",
    "anp_2023_1_raw.csv",
    "anp_2023_2_raw.csv",
]

expected_files = {
    "ANP": [ANP_DIR / name for name in expected_anp],
    "População": [POP_DIR / "populacao_raw.xls"],
    "PIB": [PIB_DIR / "pib_raw.xlsx"],
}

missing = []
for source, files in expected_files.items():
    print(f"\n{source}:")
    for file in files:
        status = "OK" if file.exists() else "AUSENTE"
        print(f"  [{status}] {file.relative_to(PROJECT_ROOT)}")
        if not file.exists():
            missing.append(file)

if missing:
    raise FileNotFoundError(
        "Há arquivos esperados ausentes. Verifique os nomes e a estrutura de pastas."
    )

print("\nTodos os arquivos esperados foram encontrados.")



ANP:
  [OK] data\bronze\anp\anp_2021_1_raw.csv
  [OK] data\bronze\anp\anp_2021_2_raw.csv
  [OK] data\bronze\anp\anp_2022_1_raw.csv
  [OK] data\bronze\anp\anp_2022_2_raw.csv
  [OK] data\bronze\anp\anp_2023_1_raw.csv
  [OK] data\bronze\anp\anp_2023_2_raw.csv

População:
  [OK] data\bronze\ibge_populacao\populacao_raw.xls

PIB:
  [OK] data\bronze\ibge_pib\pib_raw.xlsx

Todos os arquivos esperados foram encontrados.


## 3. Dependências

Execute a célula abaixo para instalar as dependências.


In [3]:
%pip install duckdb pandas openpyxl pyarrow
%pip install openpyxl
%pip install xlrd

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [4]:
import duckdb

print("DuckDB:", duckdb.__version__)
print("Pandas:", pd.__version__)


DuckDB: 1.5.5
Pandas: 2.3.3


## 4. Funções auxiliares para leitura

Os CSVs da ANP podem variar em separador e codificação. A função abaixo tenta algumas combinações comuns sem alterar os arquivos originais.


In [5]:
def read_csv_robust(path: Path) -> pd.DataFrame:
    attempts = [
        {"encoding": "utf-8-sig", "sep": None},
        {"encoding": "utf-8", "sep": None},
        {"encoding": "latin1", "sep": None},
        {"encoding": "cp1252", "sep": None},
        {"encoding": "latin1", "sep": ";"},
        {"encoding": "latin1", "sep": ","},
    ]

    last_error = None

    for params in attempts:
        try:
            df = pd.read_csv(
                path,
                engine="python",
                **params
            )

            if df.shape[1] > 1:
                return df

        except Exception as exc:
            last_error = exc

    raise ValueError(
        f"Não foi possível ler {path.name}. Erro: {last_error}"
    )

def normalize_column_names(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )
    return df


## 5. Ingestão dos arquivos ANP → `bronze.anp_raw`

In [6]:
anp_frames = []

for path in sorted(ANP_DIR.glob("anp_*_raw.csv")):
    df = read_csv_robust(path)

    # Metadados técnicos de ingestão para rastreabilidade.
    match = re.search(r"anp_(\d{4})_(\d)_raw\.csv$", path.name)
    if match:
        ano = int(match.group(1))
        semestre = int(match.group(2))
    else:
        ano = None
        semestre = None

    df["arquivo_origem"] = path.name
    df["ano_arquivo"] = ano
    df["semestre_arquivo"] = semestre
    anp_frames.append(df)

anp_raw = pd.concat(anp_frames, ignore_index=True)

print("Arquivos ANP:", len(anp_frames))
print("Linhas:", f"{len(anp_raw):,}")
print("Colunas:", len(anp_raw.columns))
display(anp_raw.head())


Arquivos ANP: 6
Linhas: 2,619,039
Colunas: 19


,Regiao - Sigla,Estado - Sigla,Municipio,Revenda,CNPJ da Revenda,Nome da Rua,Numero Rua,Complemento,Bairro,Cep,Produto,Data da Coleta,Valor de Venda,Valor de Compra,Unidade de Medida,Bandeira,arquivo_origem,ano_arquivo,semestre_arquivo
0,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA,01/01/2021,"4,599",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
1,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,ETANOL,01/01/2021,"4,199",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
2,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,GASOLINA ADITIVADA,01/01/2021,"4,799",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
3,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL,01/01/2021,"3,499",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1
4,S,RS,SAO LEOPOLDO,POSTOS DE COMBUSTIVEIS REDESINOS LTDA,01.731.418/0001-31,AVENIDA FEITORIA,4288,NaN,FEITORIA,93048-000,DIESEL S10,01/01/2021,"3,599",NaN,R$ / litro,BRANCA,anp_2021_1_raw.csv,2021,1


### 5.1. Verificação inicial dos dados ANP

Aqui fazemos somente verificações de ingestão. Regras de qualidade e transformações de negócio serão aprofundadas na Silver.


In [7]:
print("Dimensão:", anp_raw.shape)

print("\nTipos:")
display(anp_raw.dtypes.to_frame("dtype"))

print("\nNulos por coluna — top 20:")
display(
    anp_raw.isna().sum()
    .sort_values(ascending=False)
    .head(20)
    .to_frame("nulos")
)

print("\nArquivos carregados:")
display(
    anp_raw[["arquivo_origem", "ano_arquivo", "semestre_arquivo"]]
    .drop_duplicates()
    .sort_values(["ano_arquivo", "semestre_arquivo"])
)


Dimensão: (2619039, 19)

Tipos:


,dtype
Regiao - Sigla,object
Estado - Sigla,object
Municipio,object
Revenda,object
CNPJ da Revenda,object
Nome da Rua,object
Numero Rua,object
Complemento,object
Bairro,object
Cep,object



Nulos por coluna — top 20:


,nulos
Valor de Compra,2619039
Complemento,2050053
Bairro,5745
Numero Rua,951
Regiao - Sigla,0
Data da Coleta,0
ano_arquivo,0
arquivo_origem,0
Bandeira,0
Unidade de Medida,0



Arquivos carregados:


,arquivo_origem,ano_arquivo,semestre_arquivo
0,anp_2021_1_raw.csv,2021,1
334681,anp_2021_2_raw.csv,2021,2
807537,anp_2022_1_raw.csv,2022,1
1336961,anp_2022_2_raw.csv,2022,2
1715039,anp_2023_1_raw.csv,2023,1
2146615,anp_2023_2_raw.csv,2023,2


## 6. Inspeção dos arquivos Excel do IBGE

Como os arquivos Excel podem possuir mais de uma aba, primeiro listamos as abas disponíveis. O notebook usa a primeira aba por padrão, mas a seleção pode ser alterada nas variáveis abaixo após a inspeção.


In [8]:
def inspect_excel(path: Path):
    workbook = pd.ExcelFile(path)
    print(f"\nArquivo: {path.name}")
    print("Abas:")
    for i, sheet in enumerate(workbook.sheet_names):
        preview = pd.read_excel(path, sheet_name=sheet, nrows=3)
        print(f"  [{i}] {sheet!r} -> {preview.shape[1]} colunas")
    return workbook.sheet_names

pop_sheets = inspect_excel(POP_DIR / "populacao_raw.xls")
pib_sheets = inspect_excel(PIB_DIR / "pib_raw.xlsx")



Arquivo: populacao_raw.xls
Abas:
  [0] 'Municípios' -> 5 colunas

Arquivo: pib_raw.xlsx
Abas:
  [0] 'PIB dos Municípios' -> 43 colunas
  [1] 'Notas' -> 2 colunas


In [9]:
POP_SHEET = 0
PIB_SHEET = 0

print("Aba população selecionada:", pop_sheets[POP_SHEET] if isinstance(POP_SHEET, int) else POP_SHEET)
print("Aba PIB selecionada:", pib_sheets[PIB_SHEET] if isinstance(PIB_SHEET, int) else PIB_SHEET)


Aba população selecionada: Municípios
Aba PIB selecionada: PIB dos Municípios


## 7. Ingestão população → `bronze.populacao_raw`

In [10]:
populacao_raw = pd.read_excel(
    POP_DIR / "populacao_raw.xls",
    header=1,
    sheet_name=POP_SHEET
)

print("Dimensão:", populacao_raw.shape)
display(populacao_raw.head())
print("\nColunas:")
print(list(populacao_raw.columns))


Dimensão: (5603, 5)


,UF,COD. UF,COD. MUNIC,NOME DO MUNICÍPIO,POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -
0,RO,11.0,15.0,Alta Floresta D'Oeste,21494
1,RO,11.0,23.0,Ariquemes,96833
2,RO,11.0,31.0,Cabixi,5351
3,RO,11.0,49.0,Cacoal,86887
4,RO,11.0,56.0,Cerejeiras,15890



Colunas:
['UF', 'COD. UF', 'COD. MUNIC', 'NOME DO MUNICÍPIO', 'POPULAÇÃO APURADA IBGE \n- CENSO DEMOGRÁFICO 2022 E MALHA TERRITORIAL 2023 -']


## 8. Ingestão PIB → `bronze.pib_raw`

In [11]:
pib_raw = pd.read_excel(
    PIB_DIR / "pib_raw.xlsx",
    sheet_name=PIB_SHEET
)

print("Dimensão:", pib_raw.shape)
display(pib_raw.head())
print("\nColunas:")
print(list(pib_raw.columns))


Dimensão: (77965, 43)


,Ano,Código da Grande Região,Nome da Grande Região,Código da Unidade da Federação,Sigla da Unidade da Federação,Nome da Unidade da Federação,Código do Município,Nome do Município,Região Metropolitana,Código da Mesorregião,...,"Valor adicionado bruto da Indústria,\na preços correntes\n(R$ 1.000)","Valor adicionado bruto dos Serviços,\na preços correntes \n- exceto Administração, defesa, educação e saúde públicas e seguridade social\n(R$ 1.000)","Valor adicionado bruto da Administração, defesa, educação e saúde públicas e seguridade social, \na preços correntes\n(R$ 1.000)","Valor adicionado bruto total, \na preços correntes\n(R$ 1.000)","Impostos, líquidos de subsídios, sobre produtos, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto, \na preços correntes\n(R$ 1.000)","Produto Interno Bruto per capita, \na preços correntes\n(R$ 1,00)",Atividade com maior valor adicionado bruto,Atividade com segundo maior valor adicionado bruto,Atividade com terceiro maior valor adicionado bruto
0,2010,1,Norte,11,RO,Rondônia,1100015,Alta Floresta D'Oeste,NaN,1102,...,16118.534,62496.185,93244.656,241119.767,20957.111,262076.878,10731.18,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
1,2010,1,Norte,11,RO,Rondônia,1100023,Ariquemes,NaN,1102,...,287138.585,494946.267,343867.731,1199664.227,165029.553,1364693.780,15103.86,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
2,2010,1,Norte,11,RO,Rondônia,1100031,Cabixi,NaN,1102,...,3252.506,12677.210,25170.235,65400.772,4210.342,69611.114,11033.62,"Administração, defesa, educação e saúde públic...","Pecuária, inclusive apoio à pecuária",Demais serviços
3,2010,1,Norte,11,RO,Rondônia,1100049,Cacoal,NaN,1102,...,182051.537,465447.325,298454.309,1041212.374,145281.717,1186494.091,15095.15,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...
4,2010,1,Norte,11,RO,Rondônia,1100056,Cerejeiras,NaN,1102,...,19734.484,80724.991,63018.270,192454.160,29567.029,222021.189,13037.06,"Administração, defesa, educação e saúde públic...",Demais serviços,Comércio e reparação de veículos automotores e...



Colunas:
['Ano', 'Código da Grande Região', 'Nome da Grande Região', 'Código da Unidade da Federação', 'Sigla da Unidade da Federação', 'Nome da Unidade da Federação', 'Código do Município', 'Nome do Município', 'Região Metropolitana', 'Código da Mesorregião', 'Nome da Mesorregião', 'Código da Microrregião', 'Nome da Microrregião', 'Código da Região Geográfica Imediata', 'Nome da Região Geográfica Imediata', 'Município da Região Geográfica Imediata', 'Código da Região Geográfica Intermediária', 'Nome da Região Geográfica Intermediária', 'Município da Região Geográfica Intermediária', 'Código Concentração Urbana', 'Nome Concentração Urbana', 'Tipo Concentração Urbana', 'Código Arranjo Populacional', 'Nome Arranjo Populacional', 'Hierarquia Urbana', 'Hierarquia Urbana (principais categorias)', 'Código da Região Rural', 'Nome da Região Rural', 'Região rural (segundo classificação do núcleo)', 'Amazônia Legal', 'Semiárido', 'Cidade-Região de São Paulo', 'Valor adicionado bruto da Agropecu

## 9. Gravação no DuckDB

Nesta etapa criamos o banco local `database/insightfuel.duckdb` e o schema `bronze`.

As tabelas recebem os dados brutos para facilitar consultas e rastreabilidade. Os arquivos originais continuam preservados em `data/bronze`.


In [12]:
con = duckdb.connect(str(DB_PATH))

con.execute("CREATE SCHEMA IF NOT EXISTS bronze")

con.register("tmp_anp", anp_raw)
con.execute("CREATE OR REPLACE TABLE bronze.anp_raw AS SELECT * FROM tmp_anp")

con.register("tmp_populacao", populacao_raw)
con.execute("CREATE OR REPLACE TABLE bronze.populacao_raw AS SELECT * FROM tmp_populacao")

con.register("tmp_pib", pib_raw)
con.execute("CREATE OR REPLACE TABLE bronze.pib_raw AS SELECT * FROM tmp_pib")

print("Tabelas Bronze criadas com sucesso.")


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tabelas Bronze criadas com sucesso.


## 10. Log de ingestão

In [13]:
ingestion_log = pd.DataFrame([
    {
        "fonte": "ANP",
        "arquivo": p.name,
        "camada": "bronze",
        "tabela_destino": "bronze.anp_raw",
        "linhas": int(anp_raw.loc[anp_raw["arquivo_origem"] == p.name].shape[0]),
    }
    for p in sorted(ANP_DIR.glob("anp_*_raw.csv"))
] + [
    {
        "fonte": "IBGE",
        "arquivo": "populacao_raw.xlsx",
        "camada": "bronze",
        "tabela_destino": "bronze.populacao_raw",
        "linhas": int(len(populacao_raw)),
    },
    {
        "fonte": "IBGE",
        "arquivo": "pib_raw.xlsx",
        "camada": "bronze",
        "tabela_destino": "bronze.pib_raw",
        "linhas": int(len(pib_raw)),
    }
])

con.register("tmp_ingestion_log", ingestion_log)
con.execute("CREATE OR REPLACE TABLE bronze.ingestion_log AS SELECT * FROM tmp_ingestion_log")

display(con.execute("SELECT * FROM bronze.ingestion_log ORDER BY fonte, arquivo").df())



,fonte,arquivo,camada,tabela_destino,linhas
0,ANP,anp_2021_1_raw.csv,bronze,bronze.anp_raw,334681
1,ANP,anp_2021_2_raw.csv,bronze,bronze.anp_raw,472856
2,ANP,anp_2022_1_raw.csv,bronze,bronze.anp_raw,529424
3,ANP,anp_2022_2_raw.csv,bronze,bronze.anp_raw,378078
4,ANP,anp_2023_1_raw.csv,bronze,bronze.anp_raw,431576
5,ANP,anp_2023_2_raw.csv,bronze,bronze.anp_raw,472424
6,IBGE,pib_raw.xlsx,bronze,bronze.pib_raw,77965
7,IBGE,populacao_raw.xlsx,bronze,bronze.populacao_raw,5603


## 11. Testes finais da ingestão

In [14]:
tables = con.execute("""
    SELECT table_schema, table_name
    FROM information_schema.tables
    WHERE table_schema = 'bronze'
    ORDER BY table_name
""").df()

display(tables)

expected_tables = {"anp_raw", "populacao_raw", "pib_raw", "ingestion_log"}
actual_tables = set(tables["table_name"])

assert expected_tables.issubset(actual_tables), (
    f"Tabelas ausentes: {expected_tables - actual_tables}"
)

counts = con.execute("""
    SELECT
        (SELECT COUNT(*) FROM bronze.anp_raw) AS anp_linhas,
        (SELECT COUNT(*) FROM bronze.populacao_raw) AS populacao_linhas,
        (SELECT COUNT(*) FROM bronze.pib_raw) AS pib_linhas
""").df()

display(counts)

assert counts.loc[0, "anp_linhas"] > 0
assert counts.loc[0, "populacao_linhas"] > 0
assert counts.loc[0, "pib_linhas"] > 0

print("✓ Testes básicos de ingestão concluídos.")
con.close()
print("Conexão com o DuckDB encerrada.")


,table_schema,table_name
0,bronze,anp_raw
1,bronze,ingestion_log
2,bronze,pib_raw
3,bronze,populacao_raw


,anp_linhas,populacao_linhas,pib_linhas
0,2619039,5603,77965


✓ Testes básicos de ingestão concluídos.
Conexão com o DuckDB encerrada.
